# **Analyze HELM**

## Data Collator

In [1]:
%%writefile SpanMLMCollatorWithEasiness.py
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dataclasses import dataclass
from transformers import AutoTokenizer
import json
import multiprocessing

class ConfigJson():
    def __init__(self, **kwargs):
        # Assign attributes from keyword arguments
        for key, value in kwargs.items():
            setattr(self, key, value)

    @classmethod
    def from_json(cls, json_path):
        with open(json_path, "r") as file:
            data = json.load(file)
        return cls(**data)



class SpanMLMCollatorWithEasiness:

    # Define Init
    # Most of the things are just stored in the config lwk
    # Imma just pull from here
    def __init__(self, config = None, tokenizer = None, mlm_probability = None, mlm_use_span_masking = None, mlm_span_length = None, use_easiness = True):

        # Defaults
        self.mlm_probability = 0.15
        self.mlm_use_span_masking = False
        self.mlm_span_length = 3
        self.tokenizer = None

        # Override with Config (if provided)
        if (config is not None):
            if (isinstance(config, str)):
                try:
                    config = ConfigJson.from_json(config)
                except Exception as e:
                    raise ValueError(f"Blud was not a json. Either some sort of dictionary ahhh config or .json. Error: {e}")

            self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
            self.mlm_probability = config.mlm_probability
            self.mlm_use_span_masking = config.mlm_use_span_masking
            self.mlm_span_length = config.mlm_span_length

        if tokenizer is not None:
            self.tokenizer = tokenizer
        if mlm_probability is not None:
            self.mlm_probability = mlm_probability
        if mlm_use_span_masking is not None:
            self.mlm_use_span_masking = mlm_use_span_masking
        if mlm_span_length is not None:
            self.mlm_span_length = mlm_span_length


        if self.tokenizer is None:
            raise ValueError("Tokenizer is None. Either pass in the config or tokenizer")

        self.use_easiness = use_easiness

        # AI HELP:
        # Create vocab for legal words to replace
        valid_ids = [i for i in range(len(self.tokenizer)) if i not in self.tokenizer.all_special_ids]
        self.valid_vocab = torch.tensor(valid_ids)

    # __call__() function returning a DataLoader with Span_masking
    def __call__(self, data):

        # Convert List of Dictionaries into 1 large tensor
        # inside, convert extract input_ids from dictionary
        batch_input_ids = [d["input_ids"] for d in data]
        input_ids = torch.tensor(batch_input_ids)
        labels = input_ids.clone()
        easiness_score = torch.tensor([d["easiness_score"] for d in data], dtype=torch.float32)

        # Set mlm_prob
        mlm_prob = self.mlm_probability
        if (self.mlm_use_span_masking):
            mlm_prob /= self.mlm_span_length

        # Create the Masking Tensor
        masked_tensor = torch.full(input_ids.shape, mlm_prob, dtype=torch.float32)



        # =====================================================================
        # THE GUARD STEP: Protect special tokens from being masked
        # =====================================================================
        # 1. Ask the tokenizer which tokens in each row are special (CLS, SEP, PAD)
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(seq, already_has_special_tokens=True)
            for seq in input_ids.tolist()
        ]

        # 2. Convert that nested list into a PyTorch Boolean tensor
        special_tokens_map = torch.tensor(special_tokens_mask, dtype=torch.bool)

        # 3. Overwrite the probability in masked_tensor to 0.0 wherever special_tokens_map is True
        masked_tensor.masked_fill_(special_tokens_map, value=0.0)
        # =====================================================================


        # Apply Bernoulli to get 1s and 0s for the masks
        masked_tensor = torch.bernoulli(masked_tensor).bool()

        temp_clone = masked_tensor.clone()
        # Apply rolling for span masking
        if (self.mlm_use_span_masking):
            for i in range (1, self.mlm_span_length):
                rolled_tensor = torch.roll(temp_clone, shifts = i)
                rolled_tensor[:,:i] = False
                masked_tensor = masked_tensor | rolled_tensor

        # Mask all untampered tokens with -100
        labels[~masked_tensor] = -100

        # Create new Tensor w/ random values from 0-1
        type_tensor = torch.rand(input_ids.shape)

        # Create 80% mask_token_tensor
        mask_token_tensor = (type_tensor <= .8) & masked_tensor

        # Create 10% corrupted_token_tensor
        corrupted_token_tensor = (type_tensor > .8) & (type_tensor <= .9) & masked_tensor

        # Apply Mask tokens to input_ids
        input_ids[mask_token_tensor] = self.tokenizer.mask_token_id

        # LOOK HERE ##################################################

        # Apply corrupted tokens to input_ids
        num_to_replace = corrupted_token_tensor.sum().item()

        # Generate random indices for the valid_vocab_tensor
        indices = torch.randint(0,len(self.valid_vocab), (num_to_replace,))

        # Take the words form valid_vocab
        random_words = self.valid_vocab[indices]

        # Apply the words to the input_ids
        input_ids[corrupted_token_tensor] = random_words

        # ############################################################

        # Return Dictionary (based on if easiness is requested)

        if self.use_easiness:
            return {"input_ids": input_ids, "labels": labels, "easiness_score": easiness_score}

        return {"input_ids": input_ids, "labels": labels}

Writing SpanMLMCollatorWithEasiness.py


## Model

In [2]:
%%writefile model.py

##################################################
# Defines the HELM V1 architecture
# Inherited the PretrainedConfig and PreTrainedModel
# Utilizes many of the concepts found in Nvidia's 2024 nGPT architecture
# 4 -> 2 latents
# (4, 8. 16) -> (8, 12, 16) head targets
# 4 -> 8 perm heads
# Use sigmoid scaling
# Use Exclusive Attention
# Normalizes
# model_repo_id: str = "JamesResearch1216/phase06v14-final-ablation-redo"
# wandb_entity: str = "jhui16-university-of-maryland"
# wandb_project: str = "HELM-v1-10B-Run"
# wandb_name: str = "phase06v14r"
# Normalized the latent sums (@358) so that tau is less agressive (it sort of worked, but loss barely dropped)
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm"

    def __init__(

        self,

        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 16,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # Router Hyperparameters
        num_router_latents = 2,
        num_permanent_heads = 8,
        selection_threshold = 0.5,
        router_init_scale = 1.0,
        use_sigmoid_scaling = True,
        jitter_noise = 0.01,
        router_grad_clip = 0.05,
        dense_warmup_steps = 0.03,


        # Router Sparsity Hyperparameters
        sparsity_lambda = 0.01,
        sparsity_warm_up_steps = 0.05,
        head_target_min = 8,
        head_target_center = 12,
        head_target_max = 16,
        easiness_cdf_breakpoints = None,
        sparsity_slack_lo = 1.0,
        sparsity_slack_hi = 2.0,

        # Router Auxiliary Hyperparameters:
        aux_coeff_start = 0.02,
        aux_coeff_floor = 0.002,
        aux_anneal_start = 0.08,
        aux_anneal_steps = 0.25,

        # ngpt self attention and ffn hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        # Passing total step count for warm up step calculations:
        dataset_total_steps = 65000,

        **kwargs
    ):
        # General Model Hyperparameters
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization and Data Collator Hyperparameters
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # Router Hyperparameters
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.selection_threshold = selection_threshold
        self.router_init_scale = router_init_scale
        self.use_sigmoid_scaling = use_sigmoid_scaling
        self.jitter_noise = jitter_noise
        self.router_grad_clip = router_grad_clip
        self.dense_warmup_steps = int(dense_warmup_steps * dataset_total_steps)

        # Router Sparsity Hyperparameters
        self.sparsity_lambda = sparsity_lambda
        self.sparsity_warm_up_steps = int(sparsity_warm_up_steps * dataset_total_steps)
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.easiness_cdf_breakpoints = easiness_cdf_breakpoints
        self.sparsity_slack_lo = sparsity_slack_lo
        self.sparsity_slack_hi = sparsity_slack_hi

        # Router Auxiliary Hyperparameters:
        self.aux_coeff_start = aux_coeff_start
        self.aux_coeff_floor = aux_coeff_floor
        self.aux_anneal_start = int(aux_anneal_start * dataset_total_steps)
        self.aux_anneal_steps = int(aux_anneal_steps * dataset_total_steps)

        # ngpt self attention and ffn hyperparameters
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale

        super().__init__(**kwargs)



# Define Embedding Layer
class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings




# NOVEL: Multi-Latent Summary Router to decide which heads to use
class HELMMultiViewRouter(nn.Module):

    # Initialize the following:
    #   - Summary Query Matrix (q_down_proj)
    #   - Latent Importance Weights (l_i_weights)
    #   - Router_Init_Scale (tau)
    #   - Linear Router Gate (q_down_proj)
    def __init__(self, config):
        super().__init__()

        # Yoink some things from config
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_candidates = config.num_attention_heads - config.num_permanent_heads


        # Summary Query Matrix size() : [hidden_size, num_router_latents]
        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias = config.bias
        )

        # Latent Importance Weights
        # size() [num_router_latents]
        self.l_i_weights = nn.Parameter(
            torch.ones(config.num_router_latents)
        )

        # Router_Init_Scale size() : [1]
        self.tau = nn.Parameter(torch.tensor(config.router_init_scale))

        # Linear Router Gate size() : [hidden_size, num_attention_heads - num_permanent_heads]
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            config.num_attention_heads - config.num_permanent_heads,
            bias = config.bias
        )

    # Map BT_easiness -> Target # of heads
    # Problem: most of the BT_easiness_score are around .34 mark
    # We need to now center the BT easiness around the median, where a score with 0.34 should be assigned to 8/16 heads should, not 0.5
    def _easiness_to_target(self, sigmoid_scores, easiness_score):

        batch = sigmoid_scores.size(0)
        device = sigmoid_scores.device
        total_heads = self.config.num_attention_heads
        permanent_heads = self.config.num_permanent_heads

        # Get Bucket Target Values (or default to these )
        h_min = float(getattr(self.config, "head_target_min", permanent_heads + 2))
        h_ctr = float(getattr(self.config, "head_target_center", 0.5 * (h_min + total_heads)))
        h_max = float(getattr(self.config, "head_target_max", total_heads))

        # I clamped theses values before, but we'll do it here just in case
        # Size: [batch] (of easiness_score)
        easiness_score = easiness_score.to(torch.float32).view(batch).clamp(0.0,1)
        bp = getattr(self.config, "easiness_cdf_breakpoints", None)

        # Convert the breakpoints to device
        breaks = torch.as_tensor(bp, device = device, dtype=torch.float32)
        num_breaks = breaks.numel() - 1
        pos = torch.searchsorted(breaks, easiness_score, right=True).clamp(1, num_breaks)
        lo = breaks[pos - 1]; hi = breaks[pos]
        frac_in = (easiness_score - lo) / (hi - lo + 1e-8)
        q = ((pos - 1).to(torch.float32) + frac_in) / num_breaks
        q = q.clamp(0.0, 1.0)
        hard = q < 0.5
        t_hard = h_ctr + (h_max - h_ctr) * (0.5 - q) / 0.5
        t_easy = h_ctr + (h_min - h_ctr) * (q - 0.5) / 0.5
        t_total = torch.where(hard, t_hard, t_easy)

        return (t_total - permanent_heads).clamp(0.0, float(total_heads - permanent_heads))

    # Pass in only Hidden States
    # Don't pass in attention mask bc theres no attention here (duh)
    def forward(self, hidden_states, step_tensor, easiness_score):

        #################### FINALIZED LOGIC ####################

        # Write vars for cleaner code
        q_down_proj = self.q_down_proj
        l_i_weights = self.l_i_weights
        tau = self.tau
        q_up_proj = self.q_up_proj
        scale = self.scale
        self.selection_threshold = self.config.selection_threshold

        # Norm Query Matrix
        # Requires .weight since the matrix was defined before
        q_down_proj = justnorm(q_down_proj.weight, dim = 1).to(hidden_states.dtype)

        # Multiply the hidden_state by Down projection (q_down_proj)
        # Call it "scanner"
        # Size: [b, s, hidden_size] * [hidden_size * num_router_latents] = [b, s, num_router_latents]
        scanner = F.linear(hidden_states, q_down_proj)

        # Apply Softmax to entire sequences (sequence level routing)
        scanner_softmax = F.softmax(scale * scanner, dim = 1)

        # Apply Transpose to allow for dimension matching
        # [b, s, num_router_latents] -> [b, num_router_latents, s]
        scanner_softmax = scanner_softmax.transpose(1,2)

        # Create Latent Vectors (Summary of the sequence in 4 vectors)
        # Size: [b,num_router_latents,s] * [b, s, hidden_size] = [b, num_router_latents, hidden_size]
        # Use bmm (batch matrix matric product) b/c [b, n_r_l, s] * [b, s, h_s] (dims don't match up normally)
        # Could've transposed, but this is more memory efficient
        latents = torch.bmm(scanner_softmax, hidden_states)

        # Scale latents by Learnable important parameters (l_i_weights)
        # Softmax them first
        l_i_weights = F.softmax(l_i_weights, dim = 0)

        # Apply l_i_weights to latents
        # Sum the Latents together
        # Size: ([b, num_router_latents, hidden_size] * broadcast [1, num_router_latents, 1]) and sum the latents = [b, 1 (size of pooled_latents when we added them together), hidden_size]
        pooled_latents = (latents * l_i_weights.view(1, -1, 1)).sum(dim=1, keepdim = True)
        pooled_latents = justnorm(pooled_latents)

        # Normalize q_up_proj
        # Requires .weight since the matrix was defined before
        # size [total_elastic_heads, hidden_size]
        q_up_proj = justnorm(q_up_proj.weight, dim = 1).to(pooled_latents.dtype)

        # Multiply the latents by the classifer (q_up_proj)
        # Call it "class_scores"
        # Size: [b, 1, hidden_size] * [hidden_size, total_elastic_heads] = [b, 1, total_elastic_heads]
        class_scores = F.linear(pooled_latents, q_up_proj)

        # Multiply this by Tau (router_init_scale) and ngpt scaler sqrt(hidden_size), or should we???
        class_scores = class_scores * tau

        # Sigmoid Scores
        # Size: still [b, 1, total_elastic_heads], but with sigmoid scores
        sigmoid_scores = torch.sigmoid(class_scores)

        #################### FINALIZED LOGIC ENDS HERE ####################

        # Hard Mask: of 1s and 0s based on whether the sigmoid score > threshold (0.5)
        # Size: [b, 1, total_elastic_heads]
        flat_mask = (sigmoid_scores > self.selection_threshold).float()

        # Dense warmup: ensure all heads are active before dense_warmup_steps
        dense_warmup_steps = self.config.dense_warmup_steps
        in_dense_warmup = step_tensor < dense_warmup_steps
        # Use where pattern to un-mask
        # torch.ones_like (copies all metadata (device, datatype)) ; torch.ones requires you to define all metadata + shape
        flat_mask = torch.where(in_dense_warmup, torch.ones_like(flat_mask), flat_mask)

        # Telemetry hooks
        self.save_flat_mask = flat_mask.detach()
        self.save_sigmoid_scores = sigmoid_scores.detach()

        # use_sigmoid_scaling = True: router_mask = Sigmoid values and 0s (Accuracy)
        # use_sigmoid_scaling = False: router_mask = 1s and 0s (Efficiency)
        flat_mask = flat_mask * sigmoid_scores if self.config.use_sigmoid_scaling else flat_mask

        # Apply STE for Dead Router Heads during backprop
        # Must happen after sigmoid_scaling or else torch could believe the sigmoid scaling are dynamically linked
        # .detach() Ignored during Backprop (Autograd doesn't see anything with.detach(), so when backprop happens, they disappear)
        # Forward pass: flat_mask- sigmoid_scores + sigmoid_scores = flat_mask
        # Backward pass: sigmoid_scores
        flat_mask = flat_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        # Add attention dimensions: [b, 1, total_elastic_heads] -> [b, num_elastic_heads, 1, 1]
        router_mask = flat_mask.view(flat_mask.size(0), -1, 1, 1)

        # If permanent_heads are used, add the columns
        # size(): [b, num_attention_heads, 1 , 1]
        if (self.config.num_permanent_heads > 0):
            permanent_head_scores = torch.ones(
                flat_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype
            )
            router_mask = torch.cat((permanent_head_scores, router_mask), dim = 1)

        #################### LOSS CALCULATIONS ####################
        # In both previous implementations, the model could cheat by turning off all the heads to reduce loss
        # The old formulas were okay for scores passed into variant activations (softmax), but breaks at invariant activation (sigmoid)
        # We need to solve this by redefining how these losses are being calculated

        if self.training:

            # Generate the target-head count from easiness (soft PRIOR; CE can override a wrong label)
            elastic_target_num_heads =  self._easiness_to_target(sigmoid_scores, easiness_score)

            # ########## SPARSITY ##########: Ensure the correct # of heads are being activated

            # Why not count the number of heads to use > 0.5 ? Ans: > produces gradients of 0. Plus, we don't account for on edge heads (i.e .49)
            # By using the sum of the sigmoid scores along the head dimension, the gradient sees the proportion to how "almost on" heads are
            # Just think about this as how many heads should be on?
            num_head_preds = sigmoid_scores.squeeze(1).sum(-1)

            # Define our lower and upper clearances
            slack_lo = self.config.sparsity_slack_lo
            slack_hi = self.config.sparsity_slack_hi

            # WOW check this out:
            # If the computed value is < 0, then it becomes 0 (relu)
            # preds - upperbound for the over (If its less than the upperbound -> 0)
            # lowerbound - preds for the under (If its greater than the lowerbound -> 0)
            # We can define an over or under this way
            over = torch.relu(num_head_preds - (elastic_target_num_heads + slack_hi))
            under = torch.relu((elastic_target_num_heads - slack_lo) - num_head_preds)
            # Squared Sum penalty to extremely punish big changes
            raw_sparsity = (over**2 + under**2).mean()
            # Sparsity warmup
            # Denom: fixed. sparsity starts at 0 and reach 1 when step tensor reaches it
            denom = max(1, self.config.sparsity_warm_up_steps)
            # Ramp: go from 0 -> 1 starting at dense_warm_up -> sparsity_warm_up_steps + dense_warm_up
            ramp = torch.clamp((step_tensor.float() - dense_warmup_steps) / denom, 0.0, 1.0)
            self.sparsity_loss = ramp * self.config.sparsity_lambda * raw_sparsity

            # ########## SPARSITY ENDS ##########

            # ########## AUXILIARY BEGINS ##########
            # aux_loss: ensure that the router doesn't route the same head everytime (even routing)
            # This needs to be designed so its scale invariant (or else collapse will be encouraged)
            # CV^2 (coeff of variation^2) of per-head usage

            # First calculate the average sigmoid value per head in all the seqs in the batch
            # Size: [num_elastic_heads]
            head_avg_scores = sigmoid_scores.squeeze(1).mean(0)
            cv2 = head_avg_scores.var(unbiased = False) / (head_avg_scores.mean()**2 + 1e-6)

            # Yoink from config
            aux_start = self.config.aux_coeff_start
            aux_floor = self.config.aux_coeff_floor
            aux_begin = self.config.aux_anneal_start
            aux_steps = self.config.aux_anneal_steps

            # aux_frac: go from 1 -> 0 starting at aux_anneal_start -> aux_anneal_start + aux_anneal_steps
            aux_frac = torch.clamp((step_tensor.float() - aux_begin) / aux_steps, 0.0, 1.0)
            aux_coeff = aux_start + (aux_floor - aux_start) * aux_frac
            self.aux_loss = aux_coeff * cv2

            # ########## AUXILIARY ENDS ##########

        else:
            self.aux_loss = torch.tensor(0.0, device=hidden_states.device)
            self.sparsity_loss = torch.tensor(0.0, device=hidden_states.device)

        # Cast the router to matching data_type before returning:
        router_mask = router_mask.to(hidden_states.dtype)

        # Return Mask
        # [b, num_attention_heads, 1 , 1]
        return router_mask



# RoPE Class
class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.d_head = config.hidden_size // config.num_attention_heads
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None


        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            config.hidden_size * 3,
            bias = config.bias
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.hidden_size))

        # Output Matrix
        self.output = nn.Linear(
            config.hidden_size,
            config.hidden_size,
            bias = config.bias
        )

    # Configure the eval-time attention backend. Call via model.enable_efficient_inference(...).
    #   backend="flex"  : FlexAttention; set compile=True on GPU for the fused kernel (recommended).
    #   backend="gather": compact gather/scatter SDPA, no torch.compile needed.
    #   backend="dense" : compute-all-then-mask (default; what training uses).
    def set_eval_backend(self, backend="flex", compile=True):
        compile = bool(compile)
        # Only drop the cached torch.compile()'d function/block-mask builder when the
        # backend or compile flag actually changes -- resetting on every call (even when
        # nothing changed) forces a full recompilation on the very next forward pass,
        # which is silently expensive if this is called before every timed benchmark run.
        changed = (backend != getattr(self, "_eval_backend", None)
                   or compile != getattr(self, "_flex_compiled", None))
        self._eval_backend = backend
        self._flex_compiled = compile
        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(self, q, k, v, block_mask, scale):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import flex_attention
            self._flex_fn = torch.compile(flex_attention) if self._flex_compiled else flex_attention
        return self._flex_fn(q, k, v, block_mask=block_mask, scale=scale)

    def _build_block_mask(self, mask_mod, B, H, S, device):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import create_block_mask
            # compiling create_block_mask avoids materializing the full SxS mask for long sequences
            self._block_mask_fn = torch.compile(create_block_mask) if self._flex_compiled else create_block_mask
        return self._block_mask_fn(mask_mod, B, H, S, S, device=device)

    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = cast_linear(hidden_states, self.qkv)

        # Obtain Hidden Size
        batch_size, seq_len, hidden_size = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(hidden_size, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)


        eval_backend = self._eval_backend

        # Reshape q,k,v
        # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
        q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

        # Reshape q,k,v
        # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
        q = q.permute(0,2,1,3)
        k = k.permute(0,2,1,3)
        v = v.permute(0,2,1,3)


        # TRAINING / TPU MODE
        if (self.training or eval_backend == "dense"):

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Apply Attention
            # Scale by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head),
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            if router_mask is not None:
                # Apply Broadcasting Mask (expand_as() good for XLA)
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # Apply Jitter Noise to the Permanent heads during training
            if self.training and self.num_permanent_heads > 0:

                # Take the permanent heads:
                permanent_heads = context_layer[:,:self.num_permanent_heads, :, :]

                # Take the elastic heads:
                elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]

                # Apply dropout
                permanent_heads = F.dropout(permanent_heads, p = self.config.jitter_noise, training = self.training)

                # Combine back together
                context_layer = torch.cat((permanent_heads, elastic_heads),dim = 1)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # FLEX ATTENTION (for GPUs)
        elif eval_backend == "flex" and batch_size > 1:

             # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Router_mask scores, 1 or 0 or sigmoid scaling
            # [b, num_attention_heads, 1 , 1] -> [batch, num_attention_heads]
            active = (router_mask[:, :, 0, 0] > 0)

            # Boolean attention mask
            # [batch_size, 1, 1, seq_len] -> [batch, seq_len]
            key_valid = (attention_mask[:, 0, 0, :] >=0)

            # mask_mod: attend / calculate only if the head is on and its not a padding token
            def mask_mod(bi, hi, qi, ki):
                return active[bi, hi] & key_valid[bi, ki]

            # Prep the block to be passed into flex attention
            block_mask = self._build_block_mask(
                mask_mod, batch_size, self.num_attention_heads,seq_len, q.device
            )

            # Apply flex attention
            context_layer = self._flex_attn(
                q, k, v, block_mask = block_mask, scale = math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # Apply router mask to 0 the heads of the context layer
            # [batch, num attention heads, seq_len, head dim] (router_mask [batch, num_attention_heads, 1,1] was broadcasted)
            context_layer = context_layer * router_mask.expand_as(context_layer)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # Single query effieincy
        else:

            # This path only looks at batch element 0's router decisions (see below), so
            # it is only correct for batch_size == 1 -- each example's active heads are
            # data-dependent, so silently reusing example 0's mask for other examples
            # would produce wrong outputs for them instead of a loud failure.
            assert batch_size == 1, (
                f"HELMSelfAttention's 'gather' eval backend only supports batch_size == 1 "
                f"(got batch_size={batch_size}); use backend='flex' for batched inference."
            )

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # q, k, v are already [b, num_attention_heads, seq_len, d_head] from the
            # shared reshape/permute above -- no need to reshape them again here.

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor to q and k
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced.to(q_sliced.dtype) * q_sliced
            k_sliced = sqk_sliced.to(k_sliced.dtype) * k_sliced

            # Flash Attention (only for GPUs where on-the-fly splicing can exist)
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v_sliced, dim=-1)
                context_sliced = context_sliced - (context_sliced * Vn).sum(dim=-1, keepdim=True) * Vn

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims].to(context_reshaped.dtype)
            sliced_bias = None if self.output.bias is None else self.output.bias.to(context_reshaped.dtype)

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=sliced_bias)

        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention (which defines RotaryEmbeddigs) + HELMMLP
# This is 1 transformer layer
class HELMBlock(nn.Module):

    # Define the Following:
    #   - HELMMultiViewRouter
    #   - HELMSelfAttention
    #   - HELMMLP
    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    # # Define the forward pass
    def forward(self, hidden_states, attention_mask, step_tensor, easiness_score = None):
        router_mask = self.mlt_vw_rtr(hidden_states, step_tensor, easiness_score)
        aux_loss = self.mlt_vw_rtr.aux_loss
        sparsity_loss = self.mlt_vw_rtr.sparsity_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, aux_loss, sparsity_loss



# HELMModel - HELM without the head
class HELMModel(nn.Module):

    # Define the following:
    #   - HELMEmbedding
    #   - HELMBlock
    def __init__(self, config):
        super().__init__()

        # Get the ckpt_attribute
        self.use_ckpt = config.use_ckpt

        # Embedding layer
        self.embedding = HELMEmbedding(config)

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [HELMBlock(config) for _ in range(config.num_hidden_layers)]
        )


    # Forward Pass
    def forward(self, input_ids, attention_mask, current_step = None, easiness_score = None):

        # Build additive mask for SDPA fallback
        # Reshape Additive Mask to be 4D for SDPA [batch_size, 1, 1, seq_len]
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        # Convert the current_step to be infinity if null, or a tensor, or a tensor of the correct datatype if its already tensor
        # We did this because we don't want to pass in a non-tensor if we are using gradient_checkpointing
        if current_step is None:
            step_tensor = torch.tensor(float("inf"), device=input_ids.device)
        elif not isinstance(current_step, torch.Tensor):
            step_tensor = torch.tensor(current_step, device=input_ids.device)
        else:
            step_tensor = current_step

        # Pass input_ids through the input
        embeddings = self.embedding(input_ids)

        # Set Embeddings to be hidden_states
        hidden_states = embeddings.to(torch.bfloat16)

        # Accumulate aux_loss and sparsity_loss
        total_aux_loss = 0
        total_sparsity_loss = 0

        # Run Tranformer Blocks
        for block in self.blocks:
            # Use Gradient Checkpointing
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, aux_loss, sparsity_loss = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    step_tensor,
                    easiness_score,
                    use_reentrant=True if hidden_states.device.type == "xla" else False
                )
            # Or Standard Forward Pass
            else:
                hidden_states, aux_loss, sparsity_loss = block(hidden_states, attention_mask, step_tensor, easiness_score)

            total_aux_loss += aux_loss
            total_sparsity_loss += sparsity_loss

        # Return hidden state (feature extraction / context location prediction) & special losses
        # hidden_states: [b, seq_len, hidden_size]
        return hidden_states, total_aux_loss, total_sparsity_loss



# HELMModelforMaskedLM
class HELMForMaskedLM(PreTrainedModel):

    # Define the Config for the HF push_to_hub() function
    config_class = HELMConfig

    # Define the Following:
    #   - HELMModel
    #   - classifier
    #   - Head layer scaling vector
    def __init__(self, config):
        super().__init__(config)

        # Define from Config
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale

        # Define the Model
        self.model = HELMModel(config)

        # Define the head Layer
        self.classifier = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias = config.bias
        )

        # Define the head layer scaling vetor
        self.sz = nn.Parameter(torch.ones(config.vocab_size))

        # HF Function to call _init_weights() function
        self.post_init()

    # Initialize weights (pulled from ngpt model.py)
    def _init_weights(self, module):

        # If it's an nn.Linear, initialize it with the initializer_range
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        # If it's an nn.Linear, initialize it with the initializer_range also)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    # Define Function to normalize_ngpt_matrices
    # Flip every self-attention layer into an efficient eval backend for GPU inference.
    # Leave this OFF for TPU training/validation (the default "dense" path is static-shape friendly).
    #   model.eval(); model.enable_efficient_inference("flex")        # GPU, fused (recommended)
    #   model.eval(); model.enable_efficient_inference("gather")      # no torch.compile needed
    # On GPU also wrap inference in torch.compile, or pass compile=True (default) to fuse flex.
    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    # Define Function to normalize_ngpt_matrices
    def normalize_ngpt_matrices(self):

        # Define all the projection matrices to normalize
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
            "mlt_vw_rtr.q_down_proj.weight"
        )

        # Normalize every one of those mats along their dim = 1 (embedding)
        # The model's weights are transposed (for backprop) so instead of dim = 0, we do dim = 1
        with torch.no_grad():
            for name, param in self.named_parameters():
                if name.endswith(keys_to_normalize):
                    # EDIT: Instead of complete data-reassignment (danger-danger!!!), use in_place copying
                    param.copy_(justnorm(param, dim = 1, eps = 1e-12))

    # Get all necessary telemetrics & return as dict
    @torch.no_grad()
    def get_telemetry(self):

        # Define Telemetry:
        telemetry = {}

        # Iterate Through All Layers
        for i, block in enumerate(self.model.blocks):

            # ========== ROUTER ==========

            # Get the router
            router = block.mlt_vw_rtr

            # Derived / Intermediate Values:
            #   - sigmoid_scores
            #   - flat_mask
            #   - Elastic head ratio

            # sigmoid_scores
            telemetry[f"layer_{i}_sigmoid_scores"] = router.save_sigmoid_scores.float().cpu()

            # flat_mask
            telemetry[f"layer_{i}_flat_mask"] = router.save_flat_mask.float().cpu()

            # Elastic head ratio
            telemetry[f"layer_{i}_elastic_head_ratio"] = router.save_flat_mask.float().cpu().mean().item()

            # Persistent Parameters
            #   - l_i_weights
            #   - tau

            # l_i_weights (learnable importance when gather 4 latents)
            telemetry[f"layer_{i}_l_i_weights"] = router.l_i_weights.detach().cpu()

            # tau (sigmoid scaling factor)
            telemetry[f"layer_{i}_tau"] = router.tau.detach().item()

            # ============================


            # ---------- SELF ATTENTION ----------

            # sqk vector ([hidden_size] scaling vector in attn, element wise mult.)
            sqk_tensor = block.attn.sqk.detach().cpu()
            telemetry[f"layer_{i}_sqk_mean"] = sqk_tensor.mean().item()
            telemetry[f"layer_{i}_sqk_std"] = sqk_tensor.std().item()
            telemetry[f"layer_{i}_sqk_hist"] = sqk_tensor

            # ------------------------------------


            # @@@@@@@@@@ MLP @@@@@@@@@@

            # Get MLP block
            mlp = block.mlp

            # attn_alpha (post attention eigen learning rates)
            attn_alpha_tensor = mlp.attn_alpha.detach().cpu()
            telemetry[f"layer_{i}_attn_alpha_mean"] = attn_alpha_tensor.mean().item()
            telemetry[f"layer_{i}_attn_alpha_std"] = attn_alpha_tensor.std().item()
            telemetry[f"layer_{i}_attn_alpha_hist"] = attn_alpha_tensor

            # mlp_alpha (post mlp eigen learning rates)
            mlp_alpha_tensor = mlp.mlp_alpha.detach().cpu()
            telemetry[f"layer_{i}_mlp_alpha_mean"] = mlp_alpha_tensor.mean().item()
            telemetry[f"layer_{i}_mlp_alpha_std"] = mlp_alpha_tensor.std().item()
            telemetry[f"layer_{i}_mlp_alpha_hist"] = mlp_alpha_tensor

            # suv scaling vector ([intermediate_size * 2] scaling vector in u and v, element wise mult.)
            suv_tensor = mlp.suv.detach().cpu()
            telemetry[f"layer_{i}_suv_mean"] = suv_tensor.mean().item()
            telemetry[f"layer_{i}_suv_std"] = suv_tensor.std().item()
            telemetry[f"layer_{i}_suv_hist"] = suv_tensor

            # @@@@@@@@@@@@@@@@@@@@@@@@@


        # sz_tensor ([intermediate_size * 2] scaling vector in u and v, element wise mult.)
        sz_tensor = self.sz.detach().cpu()
        telemetry["lm_head_sz_mean"] = sz_tensor.mean().item()
        telemetry["lm_head_sz_std"] = sz_tensor.std().item()
        telemetry["lm_head_sz_hist"] = sz_tensor

        return telemetry



    # Forward pass
    def forward(self, input_ids, attention_mask, current_step = None, easiness_score = None):

        # Gather Context from the model
        # features: [b, seq_len, hidden_size]
        features, total_aux_loss, total_sparsity_loss = self.model(input_ids, attention_mask, current_step, easiness_score)

        # Scale / prepare sz
        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)

        # project features onto classifer
        # [b, seq_len, hidden_size] * [hidden_size, vocab_size] = [b, seq_len, vocab_size]
        unscaled_logits = cast_linear(features, self.classifier)

        # Scale the logits with sz
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits

        # Return Logits
        return logits, total_aux_loss, total_sparsity_loss




Writing model.py


## Analyzer

In [3]:
%%writefile analyze_helm.py
##################################################
# HELM inference analysis harness
#
# Answers, automatically (no eyeballing number tables):
#   A. Quality        -- CE/ppl/acc/top5/F1, overall and split by difficulty
#   B. Head census    -- which heads are always-on / always-off / genuinely dynamic
#   C. Easiness test  -- THE hypothesis test: does head count track difficulty at
#                        inference, when the model never sees the easiness label?
#   D. Layer profile  -- depth gradient in head usage
#   E. Paths          -- recurring routing motifs, co-activation, route diversity
#   F. Polarization   -- sum(sigmoid) vs count(>0.5); catches the "ratio looks high
#                        but the budget is satisfied" artifact
#   G. Timing         -- dense vs flex vs gather vs forced-dense, per sequence length
#
# Outputs: results JSON + PNG figures + a console report with explicit verdicts.
# Runs on a routed model OR the no-router baseline (router sections auto-skip).
##################################################

import os
import re
import json
import time
import random
import argparse
import warnings
from typing import List, Optional, Dict, Any, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from sklearn.metrics import accuracy_score, f1_score

from model import HELMConfig, HELMForMaskedLM
from SpanMLMCollatorWithEasiness import SpanMLMCollatorWithEasiness


DEFAULT_MODEL_REPO_ID = "JamesResearch1216/phase06v14-final-ablation-redo"
DEFAULT_DATA_REPO_ID = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
DEFAULT_TOKENIZER_PATH = "answerdotai/ModernBERT-base"
CURRICULUM_SUBSETS = {1024: "seq_1024", 2048: "seq_2048", 4096: "seq_4096"}
VALIDATION_SHARD_INDEX = {1024: 0, 2048: 1, 4096: 2}


# ============================================================
# 0. UTIL
# ============================================================

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def _j(x):
    """Make numpy/torch scalars JSON-serializable."""
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    if isinstance(x, dict):
        return {str(k): _j(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_j(v) for v in x]
    if isinstance(x, float) and (np.isnan(x) or np.isinf(x)):
        return None
    return x


# ============================================================
# 1. MODEL LOADING  (mirrors test_HELM_v1.load_helm_model)
# ============================================================

def _step_of(path):
    m = re.search(r"checkpoint-(\d+)\.pt$", os.path.basename(path))
    return int(m.group(1)) if m else -1


def load_model(checkpoint_path=None, step=None, repo_id=DEFAULT_MODEL_REPO_ID,
               hf_token=None, tokenizer_path=DEFAULT_TOKENIZER_PATH, device=None,
               config_overrides=None):
    import glob
    device = device or get_device()
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, token=hf_token)

    resolved = checkpoint_path
    if resolved is None:
        pattern = f"checkpoint-{step:06d}.pt" if step is not None else "checkpoint-*.pt"
        cands = glob.glob(os.path.join(os.getcwd(), pattern))
        if cands:
            cands.sort(key=_step_of)
            resolved = cands[-1]

    if resolved is None:
        if step is None:
            raise FileNotFoundError(
                "No local checkpoint found. Pass --checkpoint or --step so the "
                "exact weights being analyzed are unambiguous."
            )
        fn = f"checkpoint-{step:06d}.pt"
        print(f"Downloading {fn} from {repo_id} ...")
        resolved = hf_hub_download(repo_id=repo_id, filename=fn,
                                   repo_type="model", token=hf_token)

    config = HELMConfig(vocab_size=len(tokenizer), pad_token_id=tokenizer.pad_token_id)
    for k, v in (config_overrides or {}).items():
        setattr(config, k, v)

    model = HELMForMaskedLM(config)
    ckpt = torch.load(resolved, map_location="cpu", weights_only=False)
    state = ckpt["model_state"]
    state = {(k[len("module."):] if k.startswith("module.") else k): v
             for k, v in state.items()}
    model.load_state_dict(state, strict=True)
    model.to(device).eval()

    print(f"Loaded {resolved} onto {device}")
    return model, tokenizer, config, device, resolved


def has_router(model) -> bool:
    return hasattr(model.model.blocks[0], "mlt_vw_rtr")


# ============================================================
# 2. PER-SAMPLE ROUTING CAPTURE
# ============================================================
# get_telemetry() averages over the batch, which destroys the per-example
# information every path/motif question depends on. save_flat_mask is
# [b, 1, E] -- per example -- so read that directly instead.

def capture_routing(model) -> Optional[Dict[str, np.ndarray]]:
    """Returns {'mask': [B, L, E] binary, 'scores': [B, L, E] sigmoid} for the
    forward pass that just ran, or None for a no-router model."""
    if not has_router(model):
        return None
    masks, scores = [], []
    for blk in model.model.blocks:
        r = blk.mlt_vw_rtr
        masks.append(r.save_flat_mask.detach().float().cpu().squeeze(1))    # [B, E]
        scores.append(r.save_sigmoid_scores.detach().float().cpu().squeeze(1))
    return {
        "mask": torch.stack(masks, dim=1).numpy(),      # [B, L, E]
        "scores": torch.stack(scores, dim=1).numpy(),   # [B, L, E]
    }


class ForceDenseRouting:
    """Context manager: make every router emit an all-ones mask, so timing can be
    compared against a genuinely unrouted forward on identical weights."""

    def __init__(self, model):
        self.model = model
        self._orig = []

    def __enter__(self):
        if not has_router(self.model):
            return self
        for blk in self.model.model.blocks:
            r = blk.mlt_vw_rtr
            self._orig.append((r, r.forward))

            def patched(hidden_states, step_tensor, easiness_score=None, _r=r):
                b = hidden_states.size(0)
                H = _r.config.num_attention_heads
                m = torch.ones(b, H, 1, 1, device=hidden_states.device,
                               dtype=hidden_states.dtype)
                _r.aux_loss = torch.zeros((), device=hidden_states.device)
                _r.sparsity_loss = torch.zeros((), device=hidden_states.device)
                return m

            r.forward = patched
        return self

    def __exit__(self, *a):
        for r, fn in self._orig:
            r.forward = fn
        self._orig = []
        return False


# ============================================================
# 3. DATA
# ============================================================

def _is_valid_parquet(path):
    import pyarrow.parquet as pq
    try:
        pq.read_metadata(path)
        return True
    except Exception:
        return False


def load_validation_rows(seq_len, n, repo_id=DEFAULT_DATA_REPO_ID, hf_token=None,
                         seed=67, local_dir="./local_parquet_shards",
                         balance_easiness=True):
    import pyarrow.parquet as pq
    subset = CURRICULUM_SUBSETS[seq_len]
    idx = VALIDATION_SHARD_INDEX.get(seq_len, 0)
    repo_filename = f"data/{subset}/validation-{idx:05d}.parquet"
    local_path = os.path.join(local_dir, repo_filename)
    if os.path.exists(local_path) and not _is_valid_parquet(local_path):
        os.remove(local_path)
    if not os.path.exists(local_path):
        os.makedirs(local_dir, exist_ok=True)
        local_path = hf_hub_download(repo_id=repo_id, filename=repo_filename,
                                     repo_type="dataset", token=hf_token,
                                     local_dir=local_dir)
    rows = pq.read_table(local_path).to_pylist()

    if balance_easiness:
        # Deliberately span the difficulty range -- a random sample of a
        # right-skewed easiness distribution barely covers the easy tail, which
        # is exactly the region the correlation test needs.
        rows.sort(key=lambda r: r["easiness_score"])
        idx = np.linspace(0, len(rows) - 1, min(n, len(rows))).astype(int)
        return [rows[i] for i in idx]
    return random.Random(seed).sample(rows, min(n, len(rows)))


# ============================================================
# 4. EVALUATION LOOP
# ============================================================

def chunked_ce(logits, labels, vocab_size, n_chunks=8):
    lf = nn.CrossEntropyLoss(reduction="sum")
    fl = logits.reshape(-1, vocab_size)
    fb = labels.reshape(-1)
    total = fl.size(0)
    chunk = (total + n_chunks - 1) // n_chunks
    valid = (fb != -100).sum().clamp_min(1)
    s = fl.new_zeros(())
    for i in range(0, total, chunk):
        s = s + lf(fl[i:i + chunk].float(), fb[i:i + chunk])
    return s / valid


def mlm_metrics(logits, labels, loss):
    mask = labels != -100
    n = int(mask.sum().item())
    if n == 0:
        return {"accuracy": float("nan"), "top5_accuracy": float("nan"),
                "macro_f1": float("nan"), "perplexity": float("nan"),
                "num_masked_tokens": 0}
    ml = logits[mask]
    mt = labels[mask]
    preds = ml.argmax(-1).cpu().numpy()
    gold = mt.cpu().numpy()
    k = min(5, ml.size(-1))
    top5 = float((torch.topk(ml, k=k, dim=-1).indices == mt.unsqueeze(-1))
                 .any(-1).float().mean().item())
    return {
        "accuracy": float(accuracy_score(gold, preds)),
        "top5_accuracy": top5,
        "macro_f1": float(f1_score(gold, preds, average="macro", zero_division=0)),
        "perplexity": float(torch.exp(loss.detach()).item()),
        "num_masked_tokens": n,
    }


@torch.no_grad()
def evaluate(model, collator, rows, device, vocab_size, batch_size=4, backend="dense"):
    """Run the eval set, collecting per-sample metrics AND per-sample routing."""
    if has_router(model):
        model.enable_efficient_inference(backend=backend, compile=False)

    all_mask, all_scores, all_easiness = [], [], []
    per_sample = []

    for start in range(0, len(rows), batch_size):
        chunk = rows[start:start + batch_size]
        batch = collator(chunk)
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attn = torch.ones_like(input_ids, dtype=torch.long, device=device)

        logits, _, _ = model(input_ids=input_ids, attention_mask=attn)

        routing = capture_routing(model)
        if routing is not None:
            all_mask.append(routing["mask"])
            all_scores.append(routing["scores"])

        # per-sample metrics so quality can be stratified by difficulty
        for bi in range(input_ids.size(0)):
            lg = logits[bi:bi + 1]
            lb = labels[bi:bi + 1]
            if (lb != -100).sum() == 0:
                continue
            ce = chunked_ce(lg, lb, vocab_size)
            m = mlm_metrics(lg, lb, ce)
            m["ce_loss"] = float(ce.item())
            m["easiness"] = float(chunk[bi]["easiness_score"])
            if routing is not None:
                m["active_elastic"] = float(routing["mask"][bi].sum())
                m["elastic_ratio"] = float(routing["mask"][bi].mean())
            per_sample.append(m)
            all_easiness.append(m["easiness"])

    out = {"per_sample": per_sample, "easiness": np.array(all_easiness)}
    if all_mask:
        out["mask"] = np.concatenate(all_mask, axis=0)      # [N, L, E]
        out["scores"] = np.concatenate(all_scores, axis=0)  # [N, L, E]
    if has_router(model):
        model.enable_efficient_inference(backend="dense")
    return out


# ============================================================
# 5. ANALYSES
# ============================================================

def analyze_quality(per_sample) -> Dict[str, Any]:
    ez = np.array([p["easiness"] for p in per_sample])
    keys = ["ce_loss", "accuracy", "top5_accuracy", "macro_f1", "perplexity"]
    res = {"overall": {k: float(np.nanmean([p[k] for p in per_sample])) for k in keys},
           "n_samples": len(per_sample)}
    # difficulty tertiles: "hard" = low easiness
    q1, q2 = np.quantile(ez, [1 / 3, 2 / 3])
    bins = {"hard": ez <= q1, "medium": (ez > q1) & (ez <= q2), "easy": ez > q2}
    res["by_difficulty"] = {}
    for name, sel in bins.items():
        sub = [p for p, s in zip(per_sample, sel) if s]
        if not sub:
            continue
        res["by_difficulty"][name] = {
            "n": len(sub),
            "easiness_range": [float(min(p["easiness"] for p in sub)),
                               float(max(p["easiness"] for p in sub))],
            **{k: float(np.nanmean([p[k] for p in sub])) for k in keys},
        }
    res["tertile_bounds"] = [float(q1), float(q2)]
    return res


def analyze_head_census(mask, always_on=0.95, always_off=0.05) -> Dict[str, Any]:
    """Which heads are structurally on/off vs genuinely input-dependent."""
    N, L, E = mask.shape
    freq = mask.mean(axis=0)                 # [L, E] activation frequency
    cls = np.full((L, E), "dynamic", dtype=object)
    cls[freq >= always_on] = "always_on"
    cls[freq <= always_off] = "always_off"

    counts = {c: int((cls == c).sum()) for c in ["always_on", "always_off", "dynamic"]}
    # A head is only doing routing work if its activation actually varies.
    dynamic_frac = counts["dynamic"] / (L * E)
    return {
        "freq_matrix": freq,                                  # kept as array for plots
        "counts": counts,
        "dynamic_fraction": float(dynamic_frac),
        "per_layer_dynamic": [int((cls[l] == "dynamic").sum()) for l in range(L)],
        "per_layer_always_off": [int((cls[l] == "always_off").sum()) for l in range(L)],
        "per_layer_always_on": [int((cls[l] == "always_on").sum()) for l in range(L)],
        "mean_freq": float(freq.mean()),
        "std_freq": float(freq.std()),
    }


def analyze_easiness_response(mask, easiness) -> Dict[str, Any]:
    """THE hypothesis test.

    The model is never given easiness_score at inference (_easiness_to_target only
    runs under self.training). So any correlation here is the router inferring
    difficulty from content alone -- which is the entire claim of the architecture.
    """
    from scipy.stats import spearmanr, kruskal
    N, L, E = mask.shape
    total = mask.reshape(N, -1).sum(axis=1)     # active elastic heads per sample

    rho, p = spearmanr(easiness, total)
    per_layer = []
    for l in range(L):
        r_l, p_l = spearmanr(easiness, mask[:, l, :].sum(axis=1))
        per_layer.append({"layer": l, "rho": float(r_l), "p": float(p_l)})

    q1, q2 = np.quantile(easiness, [1 / 3, 2 / 3])
    g_hard = total[easiness <= q1]
    g_med = total[(easiness > q1) & (easiness <= q2)]
    g_easy = total[easiness > q2]
    try:
        kw_h, kw_p = kruskal(g_hard, g_med, g_easy)
    except Exception:
        kw_h, kw_p = float("nan"), float("nan")

    # Expected sign: harder text (LOW easiness) should use MORE heads -> negative rho.
    direction = "correct" if rho < 0 else ("inverted" if rho > 0 else "none")
    significant = bool(p < 0.05)
    return {
        "spearman_rho": float(rho), "spearman_p": float(p),
        "direction": direction, "significant": significant,
        "mean_heads_hard": float(g_hard.mean()) if len(g_hard) else None,
        "mean_heads_medium": float(g_med.mean()) if len(g_med) else None,
        "mean_heads_easy": float(g_easy.mean()) if len(g_easy) else None,
        "spread_hard_minus_easy": (float(g_hard.mean() - g_easy.mean())
                                   if len(g_hard) and len(g_easy) else None),
        "kruskal_H": float(kw_h), "kruskal_p": float(kw_p),
        "per_layer": per_layer,
        "total_heads_mean": float(total.mean()),
        "total_heads_std": float(total.std()),
        "total_heads_min": float(total.min()),
        "total_heads_max": float(total.max()),
    }


def analyze_layer_profile(mask) -> Dict[str, Any]:
    from scipy.stats import spearmanr
    N, L, E = mask.shape
    per_layer_ratio = mask.mean(axis=(0, 2))          # [L]
    per_layer_std = mask.sum(axis=2).std(axis=0)      # variability across samples
    rho, p = spearmanr(np.arange(L), per_layer_ratio)
    return {
        "per_layer_ratio": per_layer_ratio.tolist(),
        "per_layer_sample_std": per_layer_std.tolist(),
        "depth_gradient_rho": float(rho), "depth_gradient_p": float(p),
        "shallowest_ratio": float(per_layer_ratio[0]),
        "deepest_ratio": float(per_layer_ratio[-1]),
    }


def _kmeans_numpy(X, k, n_init=10, max_iter=100, seed=0):
    """Minimal k-means (numpy only). Avoids sklearn.cluster, which pulls in
    sklearn.neighbors -- a compiled submodule that's blocked/broken on some
    Windows Application-Control setups even when sklearn.metrics works fine."""
    rng = np.random.default_rng(seed)
    best_labels, best_inertia = None, np.inf
    n = X.shape[0]
    for _ in range(n_init):
        centers = X[rng.choice(n, k, replace=False)].copy()
        labels = np.zeros(n, dtype=int)
        for _ in range(max_iter):
            d = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)  # [n, k]
            new_labels = d.argmin(axis=1)
            if np.array_equal(new_labels, labels) and _ > 0:
                labels = new_labels
                break
            labels = new_labels
            for c in range(k):
                pts = X[labels == c]
                if len(pts):
                    centers[c] = pts.mean(axis=0)
        inertia = ((X - centers[labels]) ** 2).sum()
        if inertia < best_inertia:
            best_inertia, best_labels = inertia, labels
    return best_labels


def analyze_paths(mask, easiness, n_clusters=6, seed=0) -> Dict[str, Any]:
    """Do recurring routing motifs exist, and do they track difficulty?

    Route diversity also decides whether batched inference could ever be
    efficient: heterogeneous per-sequence subsets can't be grouped into
    same-shaped kernel work, so low diversity is a prerequisite, not a curiosity.
    """
    from scipy.stats import kruskal
    N, L, E = mask.shape
    flat = mask.reshape(N, -1)

    # exact-signature diversity
    sigs = {}
    for row in flat:
        key = row.astype(np.uint8).tobytes()
        sigs[key] = sigs.get(key, 0) + 1
    unique_full = len(sigs)
    top_sig_share = max(sigs.values()) / N

    # per-layer diversity -- the number that matters for grouped kernels
    per_layer_unique = []
    for l in range(L):
        s = {row.astype(np.uint8).tobytes() for row in mask[:, l, :]}
        per_layer_unique.append(len(s))

    out = {
        "unique_full_routes": unique_full,
        "unique_full_routes_frac": float(unique_full / N),
        "most_common_route_share": float(top_sig_share),
        "per_layer_unique_head_sets": per_layer_unique,
        "mean_per_layer_unique": float(np.mean(per_layer_unique)),
        "n_samples": int(N),
    }

    # pairwise Jaccard between routes (sampled, to stay cheap)
    rng = np.random.default_rng(seed)
    pairs = min(2000, N * (N - 1) // 2)
    js = []
    for _ in range(pairs):
        i, k = rng.integers(0, N, 2)
        if i == k:
            continue
        a, b = flat[i] > 0, flat[k] > 0
        union = (a | b).sum()
        js.append(((a & b).sum() / union) if union else 1.0)
    out["mean_pairwise_jaccard"] = float(np.mean(js)) if js else None

    # motif clustering
    k = min(n_clusters, max(2, N // 5))
    if N >= 2 * k and flat.std() > 0:
        lab = _kmeans_numpy(flat, k, n_init=10, seed=seed)
        groups = [easiness[lab == c] for c in range(k) if (lab == c).sum() > 0]
        try:
            H, p = kruskal(*groups) if len(groups) > 1 else (float("nan"), float("nan"))
        except Exception:
            H, p = float("nan"), float("nan")
        out["motifs"] = {
            "n_clusters": int(k),
            "cluster_sizes": [int((lab == c).sum()) for c in range(k)],
            "cluster_mean_easiness": [float(easiness[lab == c].mean())
                                      if (lab == c).sum() else None for c in range(k)],
            "cluster_mean_heads": [float(flat[lab == c].sum(axis=1).mean())
                                   if (lab == c).sum() else None for c in range(k)],
            "easiness_separation_H": float(H),
            "easiness_separation_p": float(p),
            "clusters_track_difficulty": bool(p < 0.05) if p == p else False,
        }

    # co-activation: strongest head pairs per layer
    coact = []
    for l in range(L):
        m = mask[:, l, :]
        C = (m.T @ m) / max(1, N)          # [E, E] P(i and j both on)
        np.fill_diagonal(C, 0.0)
        if C.size and C.max() > 0:
            i, jx = np.unravel_index(np.argmax(C), C.shape)
            coact.append({"layer": l, "top_pair": [int(i), int(jx)],
                          "p_joint": float(C.max())})
    out["top_coactivations"] = coact
    return out


def analyze_polarization(scores, mask) -> Dict[str, Any]:
    """sum(sigmoid) vs count(>0.5).

    The sparsity penalty constrains the SUM of sigmoid values; elastic_head_ratio
    reports the COUNT above 0.5. Those agree only when scores are polarized. If
    scores bunch near 0.5, the reported ratio can be near 1.0 while the budget is
    perfectly satisfied -- i.e. the model is soft-attenuating everything rather
    than routing. This section detects that directly.
    """
    N, L, E = scores.shape
    undecided = float(((scores > 0.4) & (scores < 0.6)).mean())
    soft_sum = scores.sum(axis=2)      # [N, L]
    hard_cnt = mask.sum(axis=2)        # [N, L]
    gap = (hard_cnt - soft_sum)
    return {
        "mean_sigmoid": float(scores.mean()),
        "std_sigmoid": float(scores.std()),
        "frac_in_undecided_band_0.4_0.6": undecided,
        "frac_below_0.1": float((scores < 0.1).mean()),
        "frac_above_0.9": float((scores > 0.9).mean()),
        "mean_soft_sum_per_layer": soft_sum.mean(axis=0).tolist(),
        "mean_hard_count_per_layer": hard_cnt.mean(axis=0).tolist(),
        "mean_count_minus_sum": float(gap.mean()),
        "per_layer_count_minus_sum": gap.mean(axis=0).tolist(),
        # near 1.0 = crisply polarized (count == sum); large gap = soft gating
        "polarization_index": float(1.0 - min(1.0, abs(gap.mean()) / max(1e-6, E * 0.5))),
    }


# ============================================================
# 6. TIMING
# ============================================================

@torch.no_grad()
def benchmark(model, collator, rows, device, seq_len, batch_sizes=(1, 4),
              backends=("dense", "flex"), reps=20, warmup=5) -> Dict[str, Any]:
    results = {}
    routed = has_router(model)

    def timed(bs, tag, ctx=None):
        chunk = rows[:bs]
        if len(chunk) < bs:
            chunk = (rows * ((bs // len(rows)) + 1))[:bs]
        batch = collator(chunk)
        ids = batch["input_ids"].to(device)
        attn = torch.ones_like(ids, dtype=torch.long, device=device)
        mgr = ctx if ctx is not None else _NullCtx()
        with mgr:
            for _ in range(warmup):
                model(input_ids=ids, attention_mask=attn)
            _sync(device)
            ts = []
            for _ in range(reps):
                t0 = time.perf_counter()
                model(input_ids=ids, attention_mask=attn)
                _sync(device)
                ts.append((time.perf_counter() - t0) * 1000.0)
        ts = np.array(ts)
        results[tag] = {"mean_ms": float(ts.mean()), "std_ms": float(ts.std()),
                        "median_ms": float(np.median(ts)), "batch_size": bs,
                        "seq_len": seq_len}
        print(f"    {tag:<34} {ts.mean():8.2f} ms  (+/- {ts.std():.2f})")

    for bs in batch_sizes:
        for be in backends:
            if routed:
                try:
                    model.enable_efficient_inference(backend=be, compile=False)
                except Exception as e:
                    print(f"    backend {be} unavailable: {e}")
                    continue
            elif be != "dense":
                continue
            timed(bs, f"bs{bs}/{be}")
        # forced-dense on identical weights = the true "no routing" reference
        if routed:
            model.enable_efficient_inference(backend="dense")
            timed(bs, f"bs{bs}/forced_all_heads", ctx=ForceDenseRouting(model))

    if routed:
        model.enable_efficient_inference(backend="dense")
    return results


class _NullCtx:
    def __enter__(self): return self
    def __exit__(self, *a): return False


# ============================================================
# 7. PLOTS
# ============================================================

def make_plots(out_dir, seq_len, head_census, layer_profile, easiness_resp,
               per_sample, polarization, mask):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    tag = f"seq{seq_len}"

    # 1. activation heatmap (layer x head)
    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(head_census["freq_matrix"], aspect="auto", cmap="viridis",
                   vmin=0, vmax=1)
    fig.colorbar(im, label="Activation frequency")
    ax.set_xlabel("Elastic head index"); ax.set_ylabel("Layer")
    ax.set_title(f"Head activation frequency ({tag}, N={mask.shape[0]})")
    fig.tight_layout(); fig.savefig(f"{out_dir}/heatmap_{tag}.png", dpi=130); plt.close(fig)

    # 2. easiness vs heads used -- the hypothesis test, visually
    ez = np.array([p["easiness"] for p in per_sample])
    hu = np.array([p.get("active_elastic", np.nan) for p in per_sample])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(ez, hu, alpha=0.6, s=22)
    if len(ez) > 2 and np.isfinite(hu).all():
        z = np.polyfit(ez, hu, 1)
        xs = np.linspace(ez.min(), ez.max(), 50)
        ax.plot(xs, np.poly1d(z)(xs), "r--",
                label=f"rho={easiness_resp['spearman_rho']:.3f}, "
                      f"p={easiness_resp['spearman_p']:.2g}")
        ax.legend()
    ax.set_xlabel("Easiness score (model never sees this)")
    ax.set_ylabel("Active elastic heads")
    ax.set_title(f"Difficulty response ({tag})")
    fig.tight_layout(); fig.savefig(f"{out_dir}/easiness_{tag}.png", dpi=130); plt.close(fig)

    # 3. layer profile
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(layer_profile["per_layer_ratio"], "o-")
    ax.set_xlabel("Layer"); ax.set_ylabel("Mean elastic ratio")
    ax.set_title(f"Depth profile ({tag}, rho={layer_profile['depth_gradient_rho']:.2f})")
    ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(f"{out_dir}/layers_{tag}.png", dpi=130); plt.close(fig)

    # 4. polarization histogram
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(polarization["_scores_flat"], bins=60)
    ax.axvline(0.5, color="r", ls="--", label="threshold")
    ax.set_xlabel("Sigmoid score"); ax.set_ylabel("Count")
    ax.set_title(f"Score polarization ({tag}) -- "
                 f"{polarization['frac_in_undecided_band_0.4_0.6']*100:.1f}% undecided")
    ax.legend()
    fig.tight_layout(); fig.savefig(f"{out_dir}/polarization_{tag}.png", dpi=130); plt.close(fig)


# ============================================================
# 8. REPORT
# ============================================================

def print_report(seq_len, quality, census, ez_resp, layers, paths, polar, timing):
    W = 78
    print("\n" + "=" * W)
    print(f"  ANALYSIS REPORT -- sequence length {seq_len}")
    print("=" * W)

    print("\n[A] QUALITY")
    o = quality["overall"]
    print(f"  N={quality['n_samples']}  CE={o['ce_loss']:.4f}  ppl={o['perplexity']:.2f}  "
          f"acc={o['accuracy']:.4f}  top5={o['top5_accuracy']:.4f}  F1={o['macro_f1']:.4f}")
    for name in ["hard", "medium", "easy"]:
        d = quality["by_difficulty"].get(name)
        if d:
            print(f"    {name:<7} (n={d['n']:>3}, ez {d['easiness_range'][0]:.2f}-"
                  f"{d['easiness_range'][1]:.2f}): CE={d['ce_loss']:.4f}  acc={d['accuracy']:.4f}")

    if ez_resp is None:
        print("\n  [no router in this checkpoint -- routing sections skipped]")
    else:
        print("\n[B] HEAD CENSUS")
        c = census["counts"]
        print(f"  always-on={c['always_on']}  always-off={c['always_off']}  "
              f"dynamic={c['dynamic']}  ({census['dynamic_fraction']*100:.1f}% dynamic)")
        print(f"  per-layer always-off: {census['per_layer_always_off']}")
        if census["dynamic_fraction"] < 0.10:
            print("  >> VERDICT: routing is near-static. Heads are structurally on/off,")
            print("     not input-dependent -- this is a learned pruning, not routing.")
        elif census["dynamic_fraction"] > 0.60:
            print("  >> VERDICT: most heads are genuinely input-dependent.")

        print("\n[C] DIFFICULTY RESPONSE  (model never sees easiness at inference)")
        print(f"  Spearman rho={ez_resp['spearman_rho']:+.4f}  p={ez_resp['spearman_p']:.2e}")
        print(f"  heads used -- hard={ez_resp['mean_heads_hard']:.2f}  "
              f"medium={ez_resp['mean_heads_medium']:.2f}  easy={ez_resp['mean_heads_easy']:.2f}")
        print(f"  hard-minus-easy spread = {ez_resp['spread_hard_minus_easy']:+.2f} heads")
        print(f"  Kruskal-Wallis H={ez_resp['kruskal_H']:.2f}  p={ez_resp['kruskal_p']:.2e}")
        if ez_resp["significant"] and ez_resp["direction"] == "correct":
            print("  >> VERDICT: PASS. The router infers difficulty from content alone")
            print("     and allocates more heads to harder text. This is the core claim.")
        elif ez_resp["significant"] and ez_resp["direction"] == "inverted":
            print("  >> VERDICT: INVERTED. Significant, but MORE heads on EASIER text.")
        else:
            print("  >> VERDICT: FAIL. No significant difficulty response at inference.")

        print("\n[D] DEPTH PROFILE")
        print(f"  ratio by layer: {[f'{r:.2f}' for r in layers['per_layer_ratio']]}")
        print(f"  depth gradient rho={layers['depth_gradient_rho']:+.3f} "
              f"(p={layers['depth_gradient_p']:.2g})  "
              f"L0={layers['shallowest_ratio']:.2f} -> L{len(layers['per_layer_ratio'])-1}="
              f"{layers['deepest_ratio']:.2f}")

        print("\n[E] ROUTE STRUCTURE")
        print(f"  unique full routes: {paths['unique_full_routes']}/{paths['n_samples']} "
              f"({paths['unique_full_routes_frac']*100:.0f}%)  "
              f"most common route = {paths['most_common_route_share']*100:.0f}% of samples")
        print(f"  unique head-sets per layer: {paths['per_layer_unique_head_sets']}")
        print(f"  mean pairwise Jaccard = {paths['mean_pairwise_jaccard']:.3f}")
        if "motifs" in paths:
            m = paths["motifs"]
            print(f"  motifs: sizes={m['cluster_sizes']}  "
                  f"mean easiness={[f'{x:.2f}' for x in m['cluster_mean_easiness']]}")
            print(f"  motif<->difficulty association p={m['easiness_separation_p']:.2e} "
                  f"-> {'TRACKS difficulty' if m['clusters_track_difficulty'] else 'no association'}")
        if paths["unique_full_routes_frac"] > 0.9:
            print("  >> NOTE: routes are almost all distinct. Batched inference cannot")
            print("     group them into same-shaped kernel work (no clustering to exploit).")

        print("\n[F] POLARIZATION  (sum-of-sigmoids vs count-above-0.5)")
        print(f"  mean sigmoid={polar['mean_sigmoid']:.3f}  "
              f"undecided(0.4-0.6)={polar['frac_in_undecided_band_0.4_0.6']*100:.1f}%  "
              f"<0.1={polar['frac_below_0.1']*100:.1f}%  >0.9={polar['frac_above_0.9']*100:.1f}%")
        print(f"  mean (hard count - soft sum) = {polar['mean_count_minus_sum']:+.2f} heads/layer")
        if polar["frac_in_undecided_band_0.4_0.6"] > 0.35:
            print("  >> VERDICT: scores are NOT polarized. The reported head ratio")
            print("     overstates real routing -- this is soft attenuation of most heads,")
            print("     and the sparsity budget can be satisfied while the ratio looks high.")
        else:
            print("  >> VERDICT: scores are polarized; ratio and budget agree.")

    if timing:
        print("\n[G] TIMING")
        for k, v in timing.items():
            print(f"  {k:<34} {v['mean_ms']:8.2f} ms  (+/- {v['std_ms']:.2f})")
        base = {k: v for k, v in timing.items() if "forced_all_heads" in k}
        for bk, bv in base.items():
            bs = bv["batch_size"]
            for k, v in timing.items():
                if f"bs{bs}/" in k and "forced" not in k:
                    sp = bv["mean_ms"] / v["mean_ms"]
                    verdict = "FASTER" if sp > 1.03 else ("slower" if sp < 0.97 else "no change")
                    print(f"  routing vs all-heads @ {k:<20} {sp:.3f}x  ({verdict})")
    print("=" * W)


# ============================================================
# 9. MAIN
# ============================================================

def main():
    ap = argparse.ArgumentParser(description="HELM inference analysis")
    ap.add_argument("--checkpoint", type=str, default=None)
    ap.add_argument("--step", type=int, default=None)
    ap.add_argument("--repo-id", type=str, default=DEFAULT_MODEL_REPO_ID)
    ap.add_argument("--data-repo-id", type=str, default=DEFAULT_DATA_REPO_ID)
    ap.add_argument("--hf-token", type=str, default=os.getenv("HF_TOKEN"))
    ap.add_argument("--seq-lens", type=int, nargs="+", default=[1024, 2048, 4096])
    ap.add_argument("--num-samples", type=int, default=100)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--backends", type=str, nargs="+", default=["dense", "flex"])
    ap.add_argument("--timing-batch-sizes", type=int, nargs="+", default=[1, 4])
    ap.add_argument("--reps", type=int, default=20)
    ap.add_argument("--skip-timing", action="store_true")
    ap.add_argument("--out-dir", type=str, default="./helm_analysis")
    ap.add_argument("--label", type=str, default="model",
                    help="Tag for this run, e.g. 'routed' or 'noroute'.")
    ap.add_argument("--seed", type=int, default=67)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)

    model, tokenizer, config, device, ckpt_path = load_model(
        checkpoint_path=args.checkpoint, step=args.step, repo_id=args.repo_id,
        hf_token=args.hf_token)
    routed = has_router(model)
    print(f"Router present: {routed}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}  "
              f"capability {torch.cuda.get_device_capability(0)}")

    collator = SpanMLMCollatorWithEasiness(tokenizer=tokenizer)

    report = {
        "label": args.label,
        "checkpoint": os.path.basename(ckpt_path),
        "routed": routed,
        "config": {
            "num_attention_heads": config.num_attention_heads,
            "num_permanent_heads": getattr(config, "num_permanent_heads", None),
            "num_hidden_layers": config.num_hidden_layers,
            "hidden_size": config.hidden_size,
            "num_router_latents": getattr(config, "num_router_latents", None),
            "head_target_min": getattr(config, "head_target_min", None),
            "head_target_center": getattr(config, "head_target_center", None),
            "head_target_max": getattr(config, "head_target_max", None),
            "use_sigmoid_scaling": getattr(config, "use_sigmoid_scaling", None),
            "use_exclusive_attention": getattr(config, "use_exclusive_attention", None),
        },
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "num_samples": args.num_samples,
        "by_seq_len": {},
    }

    for seq_len in args.seq_lens:
        print(f"\n{'#'*70}\n# Sequence length {seq_len}\n{'#'*70}")
        try:
            rows = load_validation_rows(seq_len, args.num_samples,
                                        repo_id=args.data_repo_id,
                                        hf_token=args.hf_token, seed=args.seed)
        except Exception as e:
            print(f"  could not load validation data for {seq_len}: {e}")
            continue
        print(f"  loaded {len(rows)} validation rows "
              f"(easiness {min(r['easiness_score'] for r in rows):.3f} - "
              f"{max(r['easiness_score'] for r in rows):.3f})")

        print("  running evaluation ...")
        ev = evaluate(model, collator, rows, device, config.vocab_size,
                      batch_size=args.batch_size, backend="dense")

        quality = analyze_quality(ev["per_sample"])
        census = ez_resp = layers = paths = polar = None

        if "mask" in ev:
            mask, scores, ez = ev["mask"], ev["scores"], ev["easiness"]
            census = analyze_head_census(mask)
            ez_resp = analyze_easiness_response(mask, ez)
            layers = analyze_layer_profile(mask)
            paths = analyze_paths(mask, ez, seed=args.seed)
            polar = analyze_polarization(scores, mask)
            polar["_scores_flat"] = scores.reshape(-1)
            try:
                make_plots(args.out_dir, seq_len, census, layers, ez_resp,
                           ev["per_sample"], polar, mask)
                print(f"  figures -> {args.out_dir}/*_seq{seq_len}.png")
            except Exception as e:
                print(f"  plotting failed: {e}")
            polar.pop("_scores_flat", None)
            census.pop("freq_matrix", None)   # too big for JSON; it's in the PNG

        timing = None
        if not args.skip_timing:
            print("  timing ...")
            timing = benchmark(model, collator, rows, device, seq_len,
                               batch_sizes=args.timing_batch_sizes,
                               backends=args.backends, reps=args.reps)

        print_report(seq_len, quality, census, ez_resp, layers, paths, polar, timing)

        report["by_seq_len"][str(seq_len)] = _j({
            "quality": quality, "head_census": census,
            "easiness_response": ez_resp, "layer_profile": layers,
            "paths": paths, "polarization": polar, "timing": timing,
        })

    path = os.path.join(args.out_dir, f"results_{args.label}.json")
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\nWrote {path}")
    print("Send me that JSON (and the PNGs) and I'll interpret it.")


if __name__ == "__main__":
    main()

Writing analyze_helm.py


In [4]:
!python analyze_helm.py --label base --num-samples 100 --step 65000

config.json: 100% 1.19k/1.19k [00:00<00:00, 4.05MB/s]
tokenizer_config.json: 100% 20.8k/20.8k [00:00<00:00, 55.1MB/s]
tokenizer.json: 100% 2.13M/2.13M [00:00<00:00, 44.6MB/s]
special_tokens_map.json: 100% 694/694 [00:00<00:00, 4.04MB/s]

checkpoint-065000.pt: downloading bytes:   7% 213M/3.12G [00:02<00:12, 232MB/s, 17.0MB/s  ]
checkpoint-065000.pt: downloading bytes:   9% 270M/3.12G [00:02<00:17, 167MB/s, 21.8MB/s  ]
checkpoint-065000.pt: downloading bytes:  11% 347M/3.12G [00:03<00:14, 186MB/s, 27.8MB/s  ]
checkpoint-065000.pt: downloading bytes:  50% 1.55G/3.12G [00:06<00:03, 493MB/s,  115MB/s  ]
checkpoint-065000.pt: downloading bytes:  52% 1.63G/3.12G [00:06<00:03, 452MB/s,  120MB/s  ]
checkpoint-065000.pt: downloading bytes:  66% 2.05G/3.12G [00:07<00:01, 552MB/s,  145MB/s  ]
checkpoint-065000.pt: downloading bytes:  72% 2.25G/3.12G [00:07<00:01, 633MB/s,  156MB/s  ]
checkpoint-065000.pt: downloading bytes:  90% 2.80G/3.12G [00:09<00:01, 229MB/s,  184MB/s  ]
checkpoint-065000.pt: